In [2]:
! pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 874.0/874.0 kB 6.6 MB/s eta 0:00:00


In [1]:
from tqdm import tqdm
import pandas as pd
import numpy as np
import os
import cv2

In [4]:
good_folder_path = '/content/drive/MyDrive/Sowmya /models/experiments/size_data/Sardine/Good'
bad_folder_path = '/content/drive/MyDrive/Sowmya /models/experiments/size_data/Sardine/Bad'
good_df = pd.read_excel('/content/drive/MyDrive/Sowmya /models/experiments/size_data/Sardine/good_fish_height_data.xlsx')
bad_df = pd.read_excel('/content/drive/MyDrive/Sowmya /models/experiments/size_data/Sardine/bad_fish_height_data.xlsx')
good_df.shape, bad_df.shape

((484, 2), (474, 2))

In [5]:
def delete_duplicate_filenames_rows(df):
    # Keep max index row for duplicate filenames
    df = df.loc[df.groupby('filename').apply(lambda x: x.index.max())]
    return df.reset_index(drop=True)

def extract_date(filename):
    return f"{filename[:4]}-{filename[4:6]}-{filename[6:8]}"

def process_df(df):
    # Remove duplicates, extract date, filter by height, and reset index
    df = delete_duplicate_filenames_rows(df)
    df['date'] = df['filename'].apply(extract_date)
    df = df[df['height'] >= 1].reset_index(drop=True)
    return df

# Apply processing to both bad_df and good_df
bad_df, good_df = map(process_df, [bad_df, good_df])

# Check shapes
bad_df.shape, good_df.shape


((448, 3), (467, 3))

In [6]:
min_rows = min(good_df.shape[0], bad_df.shape[0])

# Sample the dataframes to have the same number of rows
good_df_sample = good_df.sample(n=min_rows, random_state=42)
bad_df_sample = bad_df.sample(n=min_rows, random_state=42)

# Check the shapes of the sampled dataframes
good_df_sample.shape, bad_df_sample.shape

((448, 3), (448, 3))

In [7]:
good_df_sample.head()

,filename,height,date
55,20240215105859882_sardine_good.jpeg,16.3,2024-02-15
63,20240215110012026_sardine_good.jpeg,18.3,2024-02-15
33,20240215105527082_sardine_good.jpeg,17.7,2024-02-15
462,20240313112325891_sardine_good.jpeg,17.1,2024-03-13
72,20240222085528359_sardine_good.jpeg,16.4,2024-02-22


In [8]:
bad_df_sample.head()

,filename,height,date
285,20240326101301300_sardine_bad.jpeg,13.5,2024-03-26
296,20240327090941064_sardine_bad.jpeg,12.8,2024-03-27
117,20240313103256962_sardine_bad.jpeg,13.8,2024-03-13
346,20240327093333505_sardine_bad.jpeg,14.0,2024-03-27
70,20240228093056751_sardine_bad.jpeg,13.4,2024-02-28


In [9]:
from ultralytics import YOLO
model = YOLO('/content/drive/MyDrive/Sowmya /models/app model versions/segmentation/kartik/fish.pt')

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [10]:
def get_segmented_img(pred, white_background=False, index=0):
    img = cv2.cvtColor(pred.orig_img.copy(), cv2.COLOR_BGR2RGB)
    seg_img_list = []
    mask_points = pred.masks.xy[index].astype(int)

    binary_mask = np.zeros(pred.orig_shape[:2], dtype=np.uint8)
    cv2.fillPoly(binary_mask, [mask_points], 255)

    masked_img = cv2.bitwise_and(img, img, mask=binary_mask)
    x, y, w, h = cv2.boundingRect(mask_points)

    x1, y1, x2, y2 = (x, y, x+w, y+h)
    box = [x1, y1, x2, y2]

    segment_image = np.zeros((pred.orig_shape[0], pred.orig_shape[1], 3), dtype=np.uint8)
    segment_image[y:y+h, x:x+w] = masked_img[y:y+h, x:x+w]
    segment_image = segment_image[y:y+h, x:x+w]

    # white_background = False
    if white_background:
        black_mask = np.all(segment_image == [0, 0, 0], axis=-1)
        segment_image[black_mask] = [255, 255, 255]
    seg_img_list.append(segment_image)
    # return seg_img_list
    return seg_img_list

In [11]:
def circumference_area(img):
    # Load the segmented fish image
    # image = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    image = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)

    # Threshold the image to create a binary mask
    _, binary_mask = cv2.threshold(image, 1, 255, cv2.THRESH_BINARY)

    # Find contours in the binary mask
    contours, _ = cv2.findContours(binary_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    # Ensure at least one contour was found
    if len(contours) > 0:
        # Get the largest contour (presumably the fish)
        largest_contour = max(contours, key=cv2.contourArea)

        # Calculate the circumference of the contour
        circumference = cv2.arcLength(largest_contour, closed=True)

        # Calculate the area enclosed by the contour
        area = cv2.contourArea(largest_contour)

        return circumference, area
    else:
        return 0, 0

In [12]:
def get_bbox_data(yolo_results):
    """
    Compute various metrics from YOLO bounding box results.

    Args:
    - yolo_results: YOLO model results object.

    Returns:
    - bbox_data: A list of dictionaries containing the extracted metrics for each bounding box
    """
    if len(yolo_results[0].boxes.cls) == 0:
        return None

    bbox = yolo_results[0].boxes.xyxy

    image_width, image_height = yolo_results[0].orig_shape[1], yolo_results[0].orig_shape[0]
    if image_width>image_height:
      image_height, image_width = image_width, image_height

    # print(bbox[0])
    x_min, y_min, x_max, y_max = bbox.numpy().tolist()[0]

    # Compute width, height, and diagonal length
    seg_img = get_segmented_img(yolo_results[0])
    seg_img = seg_img[0]
    width, height = seg_img.shape[:2]
    if width>height:
      height, width = width, height
    diagonal_length = np.sqrt(width**2 + height**2)

    # Compute fish area and image area
    seg_img_area = width * height
    image_area = image_width * image_height
    fish_circumference, fish_area = circumference_area(seg_img)
    area_ratio = fish_area / image_area

    # Compute aspect ratio
    aspect_ratio = width / height

    # Compute diagonal to height ratio
    diagonal_to_height_ratio = diagonal_length / height

    # Normalized dimensions
    norm_width, norm_height = width / image_width, height / image_height
    norm_diagonal_length = diagonal_length / np.sqrt(image_width**2 + image_height**2)

    # Centroid of the bounding box
    centroid_x, centroid_y = x_min + width / 2, y_min + height / 2

    # Normalized centroid
    norm_centroid_x, norm_centroid_y = centroid_x / image_width, centroid_y / image_height

    # Center of the image
    center_x, center_y = image_width / 2, image_height / 2

    # Distance of the bounding box centroid to the center of the image
    distance_to_center = np.sqrt((centroid_x - center_x)**2 + (centroid_y - center_y)**2)

    # Perimeter of the bounding box
    perimeter = 2 * (width + height)

    # Compactness: perimeter^2 / area
    compactness = perimeter**2 / fish_area if fish_area > 0 else 0

    # Add the computed data to the list
    bbox_data = {
        'x_min': x_min,
        'y_min': y_min,
        'x_max': x_max,
        'y_max': y_max,
        'width': width,
        'height': height,
        'diagonal_length': diagonal_length,
        'fish_area': fish_area,
        'image_area': image_area,
        'seg_img_area': seg_img_area,
        'fish_circumference': fish_circumference,
        'area_ratio': area_ratio,
        'aspect_ratio': aspect_ratio,
        'diagonal_to_height_ratio': diagonal_to_height_ratio,
        'image_height': image_height,
        'image_width': image_width,
        'norm_width': norm_width,
        'norm_height': norm_height,
        'norm_diagonal_length': norm_diagonal_length,
        'centroid_x': centroid_x,
        'centroid_y': centroid_y,
        'norm_centroid_x': norm_centroid_x,
        'norm_centroid_y': norm_centroid_y,
        'center_x': center_x,
        'center_y': center_y,
        'distance_to_center': distance_to_center,
        'perimeter': perimeter,
        'compactness': compactness
    }

    return bbox_data

In [13]:
def convert_to_cm_dimensions(bbox_data, cm_per_pixel):
    centimeter_dict = {}
    centimeter_dict['width_cm'] = bbox_data['width'] * cm_per_pixel
    centimeter_dict['diagonal_length_cm'] = bbox_data['diagonal_length'] * cm_per_pixel
    centimeter_dict['fish_area_cm'] = bbox_data['fish_area'] * cm_per_pixel * cm_per_pixel
    centimeter_dict['image_area_cm'] = bbox_data['image_area'] * cm_per_pixel * cm_per_pixel
    centimeter_dict['seg_img_area_cm'] = bbox_data['seg_img_area'] * cm_per_pixel * cm_per_pixel
    centimeter_dict['fish_circumference_cm'] = bbox_data['fish_circumference'] * cm_per_pixel
    centimeter_dict['image_height_cm'] = bbox_data['image_height'] * cm_per_pixel
    centimeter_dict['image_width_cm'] = bbox_data['image_width'] * cm_per_pixel
    centimeter_dict['norm_width_cm'] = bbox_data['norm_width'] * cm_per_pixel
    centimeter_dict['norm_height_cm'] = bbox_data['norm_height'] * cm_per_pixel
    centimeter_dict['norm_diagonal_length_cm'] = bbox_data['norm_diagonal_length'] * cm_per_pixel
    centimeter_dict['distance_to_center_cm'] = bbox_data['distance_to_center'] * cm_per_pixel
    centimeter_dict['perimeter_cm'] = bbox_data['perimeter'] * cm_per_pixel
    return centimeter_dict

In [14]:
def yolo_predict(model, filepath):
    results = model.predict(filepath, iou=0.5, conf=0.75, verbose=False)
    return results

def process_row(row, folder_path):
    d = {'filename': row['filename'], 'height_cm': row['height']}
    filepath = os.path.join(folder_path, d['filename'])
    d['filepath'] = filepath

    # YOLO model prediction
    results = yolo_predict(model, filepath)
    bbox_data = get_bbox_data(results)

    d['fish_detected'] = 1 if bbox_data else 0

    if bbox_data is None:
        return {**d}  # Return only `d` if no fish is detected

    # Calculate cm per pixel and converted dimensions
    cm_per_pixel = d['height_cm'] / bbox_data['height']
    cm_converted_dimensions = convert_to_cm_dimensions(bbox_data, cm_per_pixel)

    # Merge all dictionaries
    combined_data = {**d, **bbox_data, **cm_converted_dimensions}
    return combined_data

def f(df, folder_path, excel_filepath):
    # Collect combined dictionaries into a list
    if excel_filepath is not None and os.path.exists(excel_filepath):
        new_df = pd.read_excel(excel_filepath)
    else:
        new_df = pd.DataFrame()

    data = []
    print("\n")
    for _, row in tqdm(df.iterrows()):
        if new_df.shape[0] > 0 and row['filename'] in new_df['filename'].values:
            # print('Skipping', row['filename'])
            continue
        new_data = process_row(row, folder_path)
        temp_df = pd.DataFrame([new_data])
        new_df = pd.concat([new_df, temp_df], ignore_index=True)
        new_df.to_excel(excel_filepath, index=False)
        data.append(new_data)

    # Convert list of dictionaries into a DataFrame
    return pd.DataFrame(data)

In [96]:
excel_filepath = '/content/drive/MyDrive/Sowmya /models/experiments/size_data/Sardine/good_fish_df_automatic.xlsx'
df_result = f(good_df_sample, good_folder_path, excel_filepath)

0it [00:00, ?it/s]

Skipping 20240215105859882_sardine_good.jpeg
Skipping 20240215110012026_sardine_good.jpeg
Skipping 20240215105527082_sardine_good.jpeg
Skipping 20240313112325891_sardine_good.jpeg
Skipping 20240222085528359_sardine_good.jpeg
Skipping 20240224091058372_sardine_good.jpeg
Skipping 20240224090650086_sardine_good.jpeg
Skipping 20240213120039042_sardine_good.jpeg
Skipping 20240222094906182_sardine_good.jpeg
Skipping 20240224093526722_sardine_good.jpeg
Skipping 20240222085611515_sardine_good.jpeg
Skipping 20240222085535861_sardine_good.jpeg
Skipping 20240313112340805_sardine_good.jpeg
Skipping 20240222085739693_sardine_good.jpeg
Skipping 20240313111652842_sardine_good.jpeg
Skipping 20240224091843099_sardine_good.jpeg
Skipping 20240302103029481_sardine_good.jpeg
Skipping 20240302102411614_sardine_good.jpeg
Skipping 20240313111628560_sardine_good.jpeg
Skipping 20240222085601583_sardine_good.jpeg
Skipping 20240213114729506_sardine_good.jpeg
Skipping 20240222085858785_sardine_good.jpeg
Skipping 2

448it [21:55,  2.94s/it]


In [97]:
df_result.head()

,filename,height_cm,filepath,fish_detected,x_min,y_min,x_max,y_max,width,height,...,image_area_cm,seg_img_area_cm,fish_circumference_cm,image_height_cm,image_width_cm,norm_width_cm,norm_height_cm,norm_diagonal_length_cm,distance_to_center_cm,perimeter_cm
0,20240222090952678_sardine_good.jpeg,17.5,/content/drive/MyDrive/Sowmya /models/experime...,1,992.264404,956.593506,3142.142822,1395.210327,440.0,2143.0,...,399.626779,62.879141,42.825627,26.654223,14.993000,0.001957,0.005362,0.004770,4.029480,42.186188
1,20240222090212356_sardine_good.jpeg,16.5,/content/drive/MyDrive/Sowmya /models/experime...,1,807.237793,852.731384,2986.301758,1363.926270,526.0,2179.0,...,343.618276,65.719826,40.835138,24.715925,13.902708,0.002169,0.005055,0.004532,2.616767,40.966039
2,20240224093345149_sardine_good.jpeg,17.0,/content/drive/MyDrive/Sowmya /models/experime...,1,850.290222,520.307373,1300.686157,2662.447021,465.0,2158.0,...,371.892816,62.272938,42.117181,25.712697,14.463392,0.001995,0.005208,0.004644,1.323462,41.326228
3,20240224091249375_sardine_good.jpeg,16.5,/content/drive/MyDrive/Sowmya /models/experime...,1,697.500610,852.124268,2834.207031,1384.134766,547.0,2138.0,...,356.923640,69.654233,40.364136,25.189897,14.169317,0.002299,0.005055,0.004548,2.268495,41.442937
4,20240302103007706_sardine_good.jpeg,18.1,/content/drive/MyDrive/Sowmya /models/experime...,1,930.975830,332.810547,1444.208130,2648.626221,526.0,2322.0,...,364.129233,74.213118,45.001916,25.442894,14.311628,0.002233,0.005545,0.004956,2.405854,44.400345


In [98]:
df_result.shape

(406, 45)

In [99]:
df_result.to_excel('/content/drive/MyDrive/Sowmya /models/experiments/size_data/Sardine/good_fish_df_manual.xlsx', index=False)

In [23]:
excel_filepath = '/content/drive/MyDrive/Sowmya /models/experiments/size_data/Sardine/bad_fish_df_automatic.xlsx'
df_result2 = f(bad_df_sample, bad_folder_path, excel_filepath)

81it [00:00, 775.83it/s]

Skipping 20240326101301300_sardine_bad.jpeg
Skipping 20240327090941064_sardine_bad.jpeg
Skipping 20240313103256962_sardine_bad.jpeg
Skipping 20240327093333505_sardine_bad.jpeg
Skipping 20240228093056751_sardine_bad.jpeg
Skipping 20240131105409725_sardine_bad.jpeg
Skipping 20240326093314013_sardine_bad.jpeg
Skipping 20240228094733603_sardine_bad.jpeg
Skipping 20240405103828114_sardine_bad.jpeg
Skipping 20240327093600176_sardine_bad.jpeg
Skipping 20240220102022510_sardine_bad.jpeg
Skipping 20240319100017811_sardine_bad.jpeg
Skipping 20240327093900685_sardine_bad.jpeg
Skipping 20240327091800722_sardine_bad.jpeg
Skipping 20240327094811401_sardine_bad.jpeg
Skipping 20240405103357587_sardine_bad.jpeg
Skipping 20240228094131503_sardine_bad.jpeg
Skipping 20240326094757198_sardine_bad.jpeg
Skipping 20240327091351346_sardine_bad.jpeg
Skipping 20240228094440341_sardine_bad.jpeg
Skipping 20240326092241425_sardine_bad.jpeg
Skipping 20240130094005438_sardine_bad.jpeg
Skipping 20240229105515833_sardi

448it [19:01,  2.55s/it]


In [24]:
df_result2.head()

,filename,height_cm,filepath,fish_detected,x_min,y_min,x_max,y_max,width,height,...,image_area_cm,seg_img_area_cm,fish_circumference_cm,image_height_cm,image_width_cm,norm_width_cm,norm_height_cm,norm_diagonal_length_cm,distance_to_center_cm,perimeter_cm
0,20240313103102165_sardine_bad.jpeg,14.0,/content/drive/MyDrive/Sowmya /models/experime...,1,755.657654,755.723389,2870.716309,1245.591675,505,2117,...,262.081993,46.754842,34.564070,21.585262,12.141710,0.001819,0.004289,0.003843,1.344496,34.679263
1,20240327091748890_sardine_bad.jpeg,12.8,/content/drive/MyDrive/Sowmya /models/experime...,1,863.394775,1055.440552,2611.402344,1452.206177,399,1735,...,326.169846,37.678478,30.466874,24.080231,13.545130,0.001603,0.003922,0.003507,2.397875,31.487262
2,20240326094056660_sardine_bad.jpeg,11.8,/content/drive/MyDrive/Sowmya /models/experime...,1,1458.034058,898.892883,3000.329834,1288.630127,379,1536,...,353.675039,34.356745,29.456397,25.075000,14.104688,0.001586,0.003615,0.003245,5.610900,29.423177
3,20240131105327009_sardine_bad.jpeg,15.7,/content/drive/MyDrive/Sowmya /models/experime...,1,961.696289,579.434875,1447.647095,2581.322510,485,1985,...,374.887629,60.225516,40.225423,25.816020,14.521511,0.002089,0.004810,0.004316,2.312934,39.072040
4,20240320100828231_sardine_bad.jpeg,12.7,/content/drive/MyDrive/Sowmya /models/experime...,1,1396.893188,817.352173,3020.724854,1136.959717,322,1623,...,366.938456,31.999618,30.520311,25.540850,14.366728,0.001372,0.003891,0.003457,5.007235,30.439310


In [25]:
df_result2.shape

(323, 45)

In [28]:
df_result2.to_excel('/content/drive/MyDrive/Sowmya /models/experiments/size_data/Sardine/bad_fish_df_manual.xlsx', index=False)

NameError: name 'df_result32' is not defined

In [2]:
good_df_auto = pd.read_excel('/content/drive/MyDrive/Sowmya /models/experiments/size_data/Sardine/good_fish_df_automatic.xlsx')
bad_df_auto = pd.read_excel('/content/drive/MyDrive/Sowmya /models/experiments/size_data/Sardine/bad_fish_df_automatic.xlsx')
good_df_auto.shape, bad_df_auto.shape

((448, 45), (448, 45))

In [3]:
good_df_manual = pd.read_excel('/content/drive/MyDrive/Sowmya /models/experiments/size_data/Sardine/good_fish_df_manual.xlsx')
# bad_df_manual = pd.read_excel('/content/drive/MyDrive/Sowmya /models/experiments/size_data/Sardine/bad_fish_df_manual.xlsx')
good_df_manual.shape

(406, 45)

In [4]:
good_df_auto.head()

,filename,height_cm,filepath,fish_detected,x_min,y_min,x_max,y_max,width,height,...,image_area_cm,seg_img_area_cm,fish_circumference_cm,image_height_cm,image_width_cm,norm_width_cm,norm_height_cm,norm_diagonal_length_cm,distance_to_center_cm,perimeter_cm
0,20240215105859882_sardine_good.jpeg,16.3,/content/drive/MyDrive/Sowmya /models/experime...,1,949.999512,438.170441,1444.094727,2470.522461,506.0,2031.0,...,385.991898,66.193570,40.349803,26.195569,14.735007,0.002212,0.004994,0.004486,2.698157,40.721910
1,20240215110012026_sardine_good.jpeg,18.3,/content/drive/MyDrive/Sowmya /models/experime...,1,603.499268,322.116882,1318.099976,2676.520020,731.0,2352.0,...,362.785877,104.083584,45.290197,25.395918,14.285204,0.003098,0.005607,0.005117,1.114710,47.975255
2,20240215105527082_sardine_good.jpeg,17.7,/content/drive/MyDrive/Sowmya /models/experime...,1,817.008667,230.234222,1453.760254,2414.274414,648.0,2169.0,...,399.071079,93.597012,44.851171,26.635685,14.982573,0.002880,0.005423,0.004933,3.164636,45.975934
3,20240313112325891_sardine_good.jpeg,17.1,/content/drive/MyDrive/Sowmya /models/experime...,1,462.403839,874.710693,2848.046875,1343.760498,485.0,2403.0,...,303.464229,59.017416,41.533355,23.226966,13.065169,0.001880,0.005239,0.004658,3.505959,41.102622
4,20240222085528359_sardine_good.jpeg,16.4,/content/drive/MyDrive/Sowmya /models/experime...,1,813.025696,871.717590,2896.575928,1425.399048,567.0,2097.0,...,366.533490,72.723090,40.576745,25.526753,14.358798,0.002415,0.005025,0.004536,2.651444,41.668670


In [5]:
bad_df_auto.head()

,filename,height_cm,filepath,fish_detected,x_min,y_min,x_max,y_max,width,height,...,image_area_cm,seg_img_area_cm,fish_circumference_cm,image_height_cm,image_width_cm,norm_width_cm,norm_height_cm,norm_diagonal_length_cm,distance_to_center_cm,perimeter_cm
0,20240326101301300_sardine_bad.jpeg,13.5,/content/drive/MyDrive/Sowmya /models/experime...,1,670.800110,819.857849,2856.918945,1376.626099,567,2199,...,225.860302,46.992156,32.405767,20.038199,11.271487,0.001896,0.004136,0.003723,1.778154,33.961801
1,20240327090941064_sardine_bad.jpeg,12.8,/content/drive/MyDrive/Sowmya /models/experime...,1,1156.335327,947.459595,2920.568115,1357.069214,424,1756,...,318.415165,39.560456,31.721581,23.792255,13.383144,0.001683,0.003922,0.003516,3.572710,31.781321
2,20240313103256962_sardine_bad.jpeg,13.8,/content/drive/MyDrive/Sowmya /models/experime...,1,1442.178101,895.619141,3037.687988,1335.279663,465,1607,...,441.925688,55.105538,34.841142,28.029371,15.766521,0.002175,0.004228,0.003836,6.523433,35.586310
3,20240327093333505_sardine_bad.jpeg,14.0,/content/drive/MyDrive/Sowmya /models/experime...,1,830.365234,677.300110,1553.828125,2833.646973,750,2158,...,252.217965,68.118628,35.869465,21.175162,11.911029,0.002650,0.004289,0.003958,2.031209,37.731233
4,20240228093056751_sardine_bad.jpeg,13.4,/content/drive/MyDrive/Sowmya /models/experime...,1,1153.594727,490.573669,1634.391113,2440.587158,481,1924,...,290.684777,44.890000,33.208632,22.732640,12.787110,0.001825,0.004105,0.003688,3.543498,33.500000


In [9]:
good_df_auto.isna().sum()

,0
filename,0
height_cm,0
filepath,0
fish_detected,0
x_min,2
y_min,2
x_max,2
y_max,2
width,2
height,2


In [10]:
bad_df_auto.isna().sum()

,0
filename,0
height_cm,0
filepath,0
fish_detected,0
x_min,0
y_min,0
x_max,0
y_max,0
width,0
height,0


In [12]:
good_df_auto['fish_detected'].unique()

array([1, 0])

In [14]:
good_df_auto[good_df_auto['fish_detected']==0]

,filename,height_cm,filepath,fish_detected,x_min,y_min,x_max,y_max,width,height,...,image_area_cm,seg_img_area_cm,fish_circumference_cm,image_height_cm,image_width_cm,norm_width_cm,norm_height_cm,norm_diagonal_length_cm,distance_to_center_cm,perimeter_cm
98,20240222094549295_sardine_good.jpeg,17.4,/content/drive/MyDrive/Sowmya /models/experime...,0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
330,20240224091555048_sardine_good.jpeg,16.2,/content/drive/MyDrive/Sowmya /models/experime...,0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [15]:
bad_df_auto['fish_detected'].unique()

array([1])

In [6]:
list(good_df_auto.columns)

['filename',
 'height_cm',
 'filepath',
 'fish_detected',
 'x_min',
 'y_min',
 'x_max',
 'y_max',
 'width',
 'height',
 'diagonal_length',
 'fish_area',
 'image_area',
 'seg_img_area',
 'fish_circumference',
 'area_ratio',
 'aspect_ratio',
 'diagonal_to_height_ratio',
 'image_height',
 'image_width',
 'norm_width',
 'norm_height',
 'norm_diagonal_length',
 'centroid_x',
 'centroid_y',
 'norm_centroid_x',
 'norm_centroid_y',
 'center_x',
 'center_y',
 'distance_to_center',
 'perimeter',
 'compactness',
 'width_cm',
 'diagonal_length_cm',
 'fish_area_cm',
 'image_area_cm',
 'seg_img_area_cm',
 'fish_circumference_cm',
 'image_height_cm',
 'image_width_cm',
 'norm_width_cm',
 'norm_height_cm',
 'norm_diagonal_length_cm',
 'distance_to_center_cm',
 'perimeter_cm']